# LangWatch Async-Native Experiment Loop with Google ADK

This notebook demonstrates `experiment.aloop` + `experiment.asubmit` — the async sibling of the threading-based `loop` / `submit` pair.

**Why async?** Google ADK's `InMemoryRunner` binds a gRPC channel to the event loop it was created on. Under the threading path each worker runs its coroutine via `asyncio.run(...)`, which spins up a *new* loop per thread and breaks that binding (`"Future attached to a different loop"`). The async-native path keeps every item on the caller's event loop, so singletons and factories that open expensive connections once stay usable across items.

**Requirements**:
```bash
pip install langwatch google-adk openinference-instrumentation-google-adk
```
And set `LANGWATCH_API_KEY` + `GOOGLE_API_KEY` in your environment.

In [ ]:
import langwatch
from openinference.instrumentation.google_adk import GoogleADKInstrumentor

langwatch.login()
langwatch.setup(instrumentors=[GoogleADKInstrumentor()])

## Create the agent and a shared runner

The runner is created **once**, on this notebook's event loop. Under the threading path, sharing it across concurrent items would raise `"Future attached to a different loop"`. With `aloop` / `asubmit`, every item runs on this same loop so the runner stays valid.

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types


def get_weather(city: str) -> dict:
    forecasts = {
        "new york": "Sunny, 25C",
        "london": "Cloudy, 18C",
        "tokyo": "Rainy, 22C",
        "berlin": "Windy, 16C",
        "sao paulo": "Humid, 27C",
    }
    city = city.lower().strip()
    if city in forecasts:
        return {"status": "ok", "report": forecasts[city]}
    return {"status": "error", "message": f"no forecast for {city!r}"}


agent = Agent(
    name="weather_agent",
    model="gemini-2.0-flash-exp",
    description="Replies with a short weather report when asked about a city.",
    instruction="Use the get_weather tool, then answer briefly.",
    tools=[get_weather],
)

runner = InMemoryRunner(agent=agent, app_name="experiment-async-adk")

## Run the experiment with `aloop` + `asubmit`

Each item awaits the shared `runner` concurrently. `concurrency=4` bounds the number of in-flight tasks; the rest queue on an `asyncio.Semaphore`. Sync callables passed to `asubmit` would be offloaded to a worker thread automatically, but here every task is `async`.

In [ ]:
cities = [
    "New York",
    "London",
    "Tokyo",
    "Berlin",
    "Sao Paulo",
    "New York",
    "London",
    "Tokyo",
    "Berlin",
    "Sao Paulo",
]

experiment = langwatch.experiment.init("async-adk-example")


async def ask(city: str, *, item_index: int) -> str:
    session_id = f"item-{item_index}"
    await runner.session_service.create_session(
        app_name="experiment-async-adk", user_id="user", session_id=session_id
    )
    async for event in runner.run_async(
        user_id="user",
        session_id=session_id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=f"What is the weather in {city}?")],
        ),
    ):
        if event.is_final_response():
            # ADK's is_final_response() can fire for events whose content is
            # None (e.g. state-delta events), so guard every step before
            # indexing parts[0].text.
            content = getattr(event, "content", None)
            parts = getattr(content, "parts", None) or []
            text = getattr(parts[0], "text", None) if parts else None
            response = text.strip() if text else ""
            experiment.log_response(response)
            return response
    return ""


item_index = 0
async for city in experiment.aloop(cities, concurrency=4):
    experiment.asubmit(ask, city, item_index=item_index)
    item_index += 1